# Limpieza y unión del catálogo StreamView

Este notebook carga las fuentes oficiales, revisa su estructura, aplica limpieza verificable, homologa el esquema y exporta un catálogo integrado. No incluye EDA, gráficos ni análisis de negocio.

Regla de duplicados aplicada: se eliminan únicamente filas completamente idénticas. Los `show_id` repetidos entre Movies y TV Shows se conservan porque corresponden a contenidos distintos.

In [151]:
import pandas as pd

movies = pd.read_csv("/content/netflix_movies_detailed_up_to_2025.csv")
tv_shows = pd.read_csv("/content/netflix_tv_shows_detailed_up_to_2025.csv")

### Carga Inicial de Datos

En esta sección, cargamos los conjuntos de datos originales de películas (`netflix_movies_detailed_up_to_2025.csv`) y programas de televisión (`netflix_tv_shows_detailed_up_to_2025.csv`) en DataFrames de pandas para su posterior procesamiento y análisis.

## Funciones Auxiliares y Configuración de Rutas

Aquí se definen las rutas de los archivos de origen y de destino, junto con una función para calcular el hash de los archivos, utilizada para verificar su integridad. La verificación del hash se realiza al inicio para asegurar que los archivos originales no han sido modificados antes del procesamiento.

In [152]:
import hashlib
from pathlib import Path

# Rutas de los archivos de entrada y salida
movies_path = "/content/netflix_movies_detailed_up_to_2025.csv"
tv_path = "/content/netflix_tv_shows_detailed_up_to_2025.csv"
output_path = Path("/content/clean_netflix_catalog.csv")

# Función para calcular el hash de un archivo
def file_hash(filepath):
    hasher = hashlib.md5()
    with open(filepath, 'rb') as f:
        while True:
            chunk = f.read(4096)
            if not chunk:
                break
            hasher.update(chunk)
    return hasher.hexdigest()

# Calcular los hashes de los archivos originales antes de cualquier modificación
# Esto es para verificar la integridad de los datos fuente.
raw_hashes_before = {
    'movies': file_hash(movies_path),
    'tv_shows': file_hash(tv_path),
}

print("Rutas de archivos y función de hash definidas.")
print(f"Hashes de archivos originales: {raw_hashes_before}")

Rutas de archivos y función de hash definidas.
Hashes de archivos originales: {'movies': '2d26447c66dd52b9a18a7e52d7a9499f', 'tv_shows': 'b044d6d636d54de5535ff5912c09a98b'}


### Inspección Inicial de Datos

Antes de cualquier transformación, es crucial realizar una inspección rápida de los primeros registros de ambos DataFrames. Esto nos permite entender la estructura de los datos, identificar las columnas presentes y observar posibles problemas iniciales como formatos inesperados o valores atípicos.

In [153]:
movies.head(5)

,show_id,type,title,director,cast,country,date_added,release_year,rating,duration,genres,language,description,popularity,vote_count,vote_average,budget,revenue
0,10192,Movie,Shrek Forever After,Mike Mitchell,"Mike Myers, Eddie Murphy, Cameron Diaz, Antoni...",United States of America,2010-05-16,2010,6.380,NaN,"Comedy, Adventure, Fantasy, Animation, Family",en,A bored and domesticated Shrek pacts with deal...,203.893,7449,6.380,165000000,752600867
1,27205,Movie,Inception,Christopher Nolan,"Leonardo DiCaprio, Joseph Gordon-Levitt, Ken W...","United Kingdom, United States of America",2010-07-15,2010,8.369,NaN,"Action, Science Fiction, Adventure",en,"Cobb, a skilled thief who commits corporate es...",156.242,37119,8.369,160000000,839030630
2,12444,Movie,Harry Potter and the Deathly Hallows: Part 1,David Yates,"Daniel Radcliffe, Emma Watson, Rupert Grint, T...","United Kingdom, United States of America",2010-11-17,2010,7.744,NaN,"Adventure, Fantasy",en,"Harry, Ron and Hermione walk away from their l...",121.191,19327,7.744,250000000,954305868
3,38757,Movie,Tangled,"Byron Howard, Nathan Greno","Mandy Moore, Zachary Levi, Donna Murphy, Ron P...",United States of America,2010-11-24,2010,7.600,NaN,"Animation, Family, Adventure",en,"Feisty teenager Rapunzel, who has long and mag...",111.762,11638,7.600,260000000,592461732
4,10191,Movie,How to Train Your Dragon,"Chris Sanders, Dean DeBlois","Jay Baruchel, Gerard Butler, Craig Ferguson, A...",United States of America,2010-03-18,2010,7.800,NaN,"Fantasy, Adventure, Animation, Family",en,As the son of a Viking leader on the cusp of m...,110.044,13259,7.800,165000000,494879471


In [154]:
tv_shows.head(5)

,show_id,type,title,director,cast,country,date_added,release_year,rating,duration,genres,language,description,popularity,vote_count,vote_average
0,33238,TV Show,Running Man,안재철,"Yoo Jae-suk, Jee Seok-jin, Kim Jong-kook, Haha...",South Korea,2010-07-11,2010,8.241,1 Seasons,"Comedy, Reality",ko,A reality and competition show where members a...,1929.898,187,8.241
1,32415,TV Show,Conan,NaN,"Conan O'Brien, Andy Richter",United States of America,2010-11-08,2010,7.035,1 Seasons,"Talk, Comedy, News",en,A late night television talk show hosted by C...,1670.580,229,7.035
2,37757,TV Show,MasterChef Greece,NaN,NaN,Greece,2010-10-03,2010,5.600,1 Seasons,Reality,el,MasterChef Greece is a Greek competitive cooki...,1317.092,6,5.600
3,75685,TV Show,Prostřeno!,NaN,"Václav Vydra, Jana Boušková",Czech Republic,2010-03-01,2010,6.500,1 Seasons,Reality,cs,The knives (and forks) are out as a group of s...,1095.776,6,6.500
4,33847,TV Show,The Talk,NaN,"Amanda Kloots, Jerry O'Connell, Akbar Gbaja-Bi...","United States of America, Ireland",2010-10-18,2010,3.400,1 Seasons,Talk,en,A panel of well-known news and entertainment p...,712.070,12,3.400


A traves de la siguiente visualización, podemos apreciar la distribución de nulos.

In [155]:
display(pd.DataFrame({
    'nulos_movies': movies.isna().sum(),
    'nulos_tv_shows': tv_shows.isna().sum(),
}))

,nulos_movies,nulos_tv_shows
budget,0,NaN
cast,204,1157.0
country,466,1797.0
date_added,0,0.0
description,132,3206.0
director,132,10965.0
duration,16000,0.0
genres,107,974.0
language,0,0.0
popularity,0,0.0


### Análisis de Nulos y Columnas Únicas

Para entender la completitud de nuestros datos, calculamos la cantidad de valores nulos en cada columna de ambos DataFrames. Además, identificamos las columnas que son exclusivas de cada conjunto de datos, lo cual es fundamental para el proceso de homologación de esquemas antes de la unión.



> Como observación, tanto en budget como en revenue, nulos_tv_shows los muestra como NaN, y esto es porque en realidad, aquellas columnas no existen en el dataset.



In [156]:
print('Columnas exclusivas de Movies:', sorted(set(movies.columns) - set(tv_shows.columns)))
print('Columnas exclusivas de TV Shows:', sorted(set(tv_shows.columns) - set(movies.columns)))

Columnas exclusivas de Movies: ['budget', 'revenue']
Columnas exclusivas de TV Shows: []


### Resumen de Columnas Exclusivas o con Datos Exclusivos

Basándonos en la inspección de los datasets originales:

*   **`budget`** y **`revenue`**:
    > Estas columnas pertenecen **exclusivamente a las 'Movies'**, ya que solo existen en el dataset original de películas. Para los 'TV Shows', se añadieron como nulas para la homologación.

*   **`duration`**:
    > Aunque la columna `duration` está presente en ambos datasets originales, su contenido significativo pertenece **exclusivamente a los 'TV Shows'**. Para las 'Movies', esta columna estaba completamente vacía en el dataset original, lo que la hacía irrelevante antes de la imputación.

Esto nos muestra que movies solo tiene dos columnas extra, por lo que es posible una unión.

Se crea funcion capaz de transformar vacios a nulos.
Esto con el fin de estandarizar ambos datasets.


In [157]:
def normalize_text_columns(dataframe):
    cleaned = dataframe.copy()
    # Itera sobre todas las columnas de tipo 'object' (cadenas de texto)
    for column in cleaned.select_dtypes(include='object').columns:
        # Elimina espacios en blanco al inicio y final, y reemplaza cadenas vacías por nulos
        cleaned[column] = cleaned[column].str.strip().replace('', pd.NA)
    return cleaned

movies_clean = normalize_text_columns(movies)
tv_shows_clean = normalize_text_columns(tv_shows)

Para asegurar los datos, se sigue comprobando si existen duplicados
que puedan afectar al trabajo

##Comprobacion duplicados (Se omite por falta de resultados)

In [158]:
# Solo se eliminan duplicados de filas completas, según DATA_RULES.md (reglas de datos).
movies_clean = movies_clean.drop_duplicates().copy()
tv_shows_clean = tv_shows_clean.drop_duplicates().copy()

### Homologación de Esquemas y Relleno de Columnas Exclusivas

Dado que los conjuntos de datos de películas y programas de televisión no tienen exactamente las mismas columnas (ej. `budget` y `revenue` son exclusivos de películas), esta sección se encarga de armonizar los esquemas. Se añaden las columnas faltantes a `tv_shows_clean` y se reordenan las columnas para asegurar que ambos DataFrames tengan la misma estructura y orden antes de la concatenación.

In [159]:
# Las variables exclusivas de Movies ('budget', 'revenue') se mantienen
# como nulas para TV Shows, ya que estas columnas no aplican para TV Shows.
for column in ['budget', 'revenue']:
    if column not in tv_shows_clean.columns:
        tv_shows_clean[column] = pd.NA

# Asegura que las columnas de tv_shows_clean estén en el mismo orden que movies_clean
# Esto es crucial para la concatenación posterior.
column_order = list(movies_clean.columns)
tv_shows_clean = tv_shows_clean.reindex(columns=column_order)

### Conversión de Tipos de Datos

Para garantizar la correcta manipulación y análisis de los datos, convertimos las columnas a sus tipos de datos apropiados (ej. `date_added` a `datetime`, `release_year` a numérico entero, `vote_count` a numérico entero, y otras columnas numéricas a `float`). Esto ayuda a evitar errores en cálculos y validaciones futuras.

In [160]:
for dataframe in [movies_clean, tv_shows_clean]:
    dataframe['show_id'] = dataframe['show_id'].astype('string')
    dataframe['date_added'] = pd.to_datetime(dataframe['date_added'], errors='raise')
    dataframe['release_year'] = pd.to_numeric(dataframe['release_year'], errors='raise').astype('Int64')
    dataframe['vote_count'] = pd.to_numeric(dataframe['vote_count'], errors='raise').astype('Int64')
    for column in ['rating', 'popularity', 'vote_average', 'budget', 'revenue']:
        dataframe[column] = pd.to_numeric(dataframe[column], errors='raise')

In [161]:
for dataframe in [movies_clean, tv_shows_clean]:
    dataframe['show_id'] = dataframe['show_id'].astype('string')
    dataframe['date_added'] = pd.to_datetime(dataframe['date_added'], errors='coerce') # Cambiado 'raise' a 'coerce'
    dataframe['release_year'] = pd.to_numeric(dataframe['release_year'], errors='raise').astype('Int64')
    dataframe['vote_count'] = pd.to_numeric(dataframe['vote_count'], errors='raise').astype('Int64')
    for column in ['rating', 'popularity', 'vote_average', 'budget', 'revenue']:
        dataframe[column] = pd.to_numeric(dataframe[column], errors='raise')

In [162]:
print("Nulos en 'date_added' después de la conversión a datetime:")
print(f"Movies clean: {movies_clean['date_added'].isna().sum()} nulos")
print(f"TV Shows clean: {tv_shows_clean['date_added'].isna().sum()} nulos")

if movies_clean['date_added'].isna().sum() > 0 or tv_shows_clean['date_added'].isna().sum() > 0:
    print("ADVERTENCIA: Se encontraron nulos en 'date_added' después de la conversión a datetime. Revisar los datos originales para posibles formatos de fecha inconsistentes.")
else:
    print("No se encontraron nulos en 'date_added' después de la conversión a datetime.")

Nulos en 'date_added' después de la conversión a datetime:
Movies clean: 0 nulos
TV Shows clean: 0 nulos
No se encontraron nulos en 'date_added' después de la conversión a datetime.


In [163]:
print(f'Movies: {len(movies)} -> {len(movies_clean)} filas')
print(f'TV Shows: {len(tv_shows)} -> {len(tv_shows_clean)} filas')

Movies: 16000 -> 16000 filas
TV Shows: 16000 -> 16000 filas


### Resumen de Filas Limpias

Después de aplicar las operaciones de limpieza y normalización, verificamos la cantidad de filas en cada DataFrame (`movies_clean` y `tv_shows_clean`) para asegurar que no se hayan perdido registros inesperadamente y que el proceso de limpieza haya sido exitoso en términos de volumen de datos.

## Posterior a la comprobacion, se unen ambos datasets ya limpiados

In [164]:
catalogo = pd.concat([movies_clean, tv_shows_clean], ignore_index=True)

# Eliminar la columna 'description' como se decidió.
catalogo = catalogo.drop(columns=['description'])

# Validaciones previas a la exportación para asegurar la calidad de los datos.
# 1. Verifica que la columna 'type' solo contenga 'Movie' o 'TV Show'.
assert set(catalogo['type'].dropna().unique()) == {'Movie', 'TV Show'}
# 2. Verifica que no existan filas completamente duplicadas.
assert not catalogo.duplicated().any(), 'Persisten filas completamente duplicadas.'
# 3. Verifica que los TV Shows no tengan valores en 'budget' o 'revenue'.
assert catalogo.loc[catalogo['type'].eq('TV Show'), ['budget', 'revenue']].isna().all().all()
# 4. Verifica que los archivos fuente originales no hayan sido modificados desde la carga inicial.
assert raw_hashes_before == {
    'movies': file_hash(movies_path),
    'tv_shows': file_hash(tv_path),
}, 'Las fuentes originales fueron modificadas.'

# Formatea la columna 'date_added' a un formato de fecha estándar (YYYY-MM-DD).
catalogo['date_added'] = catalogo['date_added'].dt.strftime('%Y-%m-%d')

# Crea el directorio de salida si no existe.
output_path.parent.mkdir(parents=True, exist_ok=True)

# Exporta el DataFrame 'catalogo' a un archivo CSV.
catalogo.to_csv(output_path, index=False)

print(f'Archivo exportado: {output_path}')
print(f'Filas: {len(catalogo):,}')

# Muestra el recuento de 'Movies' y 'TV Shows' en el catálogo final.
display(catalogo['type'].value_counts().rename_axis('type').to_frame('filas'))

Archivo exportado: /content/clean_netflix_catalog.csv
Filas: 32,000


,filas
type,
Movie,16000
TV Show,16000


### Verificación Final del Catálogo Exportado

Una vez que el catálogo unificado ha sido exportado a un archivo CSV, realizamos una verificación final. Volvemos a cargar el archivo exportado y realizamos varias aserciones para confirmar que los datos se guardaron correctamente, manteniendo la integridad, la unicidad y los tipos de datos esperados. Esto asegura que el archivo final esté listo para su uso.

In [165]:
catalogo_verificado = pd.read_csv(output_path)

assert len(catalogo_verificado) == len(catalogo)
assert set(catalogo_verificado['type'].unique()) == {'Movie', 'TV Show'}
assert not catalogo_verificado.duplicated().any()

print('Validación final completada correctamente.')
display(catalogo_verificado.info())

Validación final completada correctamente.
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 32000 entries, 0 to 31999
Data columns (total 17 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   show_id       32000 non-null  int64  
 1   type          32000 non-null  object 
 2   title         32000 non-null  object 
 3   director      20903 non-null  object 
 4   cast          30639 non-null  object 
 5   country       29737 non-null  object 
 6   date_added    32000 non-null  object 
 7   release_year  32000 non-null  int64  
 8   rating        32000 non-null  float64
 9   duration      16000 non-null  object 
 10  genres        30919 non-null  object 
 11  language      32000 non-null  object 
 12  popularity    32000 non-null  float64
 13  vote_count    32000 non-null  int64  
 14  vote_average  32000 non-null  float64
 15  budget        16000 non-null  float64
 16  revenue       16000 non-null  float64
dtypes: float64(5), int64(3), o

None

Filtrar el DataFrame para solo incluir type == 'Movie' antes de calcular promedios o realizar análisis financieros que dependan de budget y revenue.


## Análisis y Estrategias de Imputación para Columnas con Nulos

Identificaremos las columnas con valores nulos en el DataFrame `catalogo` y, para cada una, exploraremos su naturaleza y discutiremos posibles técnicas de imputación o estandarización.

In [166]:
# Mostrar la cantidad de nulos por columna en el catálogo unificado
nulos_catalogo = catalogo.isna().sum()
nulos_catalogo = nulos_catalogo[nulos_catalogo > 0].sort_values(ascending=False)

print("Columnas con valores nulos en el catálogo:")
display(nulos_catalogo.to_frame('Cantidad de Nulos'))

Columnas con valores nulos en el catálogo:


,Cantidad de Nulos
revenue,16000
budget,16000
duration,16000
director,11097
country,2263
cast,1361
genres,1081


### Columna: `duration` (Revisión Detallada)

La columna `duration` presenta un número considerable de valores nulos. Esto se debe a que, en el dataset original de películas (`movies`), esta columna estaba **completamente vacía para todas las películas**. Es decir, el campo `duration` no contenía información alguna en el archivo de origen de películas, a pesar de que la columna `duration` como tal sí estaba presente en la estructura del dataset de películas. Por lo tanto, en términos de *datos significativos*, la columna `duration` **pertenecía exclusivamente a los programas de TV** (`tv_shows`) en los datasets originales, donde sí contiene información relevante (el número de temporadas) y no presentaba nulos. Los nulos en `duration` provienen, entonces, exclusivamente del dataset de películas y representan una ausencia total de dato útil en el origen para este tipo de contenido.

**Estrategias de Imputación Sugeridas y Decisión Final:**
1.  **Conservación de Nulos para Películas:** Para las películas, se ha decidido **mantener los valores nulos (`NaN`) en la columna `duration`**. Esto refleja que la información no estaba disponible en el dataset original y evita la imputación de un valor que podría interpretarse como una duración real. La ausencia de este dato es, en sí misma, información.
2.  **Mantenimiento de Valores Originales para TV Shows:** Para los programas de TV, se conservarán los valores originales de `duration` (ej. '1 Season', '2 Seasons'), ya que esta columna ya contenía datos significativos y sin nulos en su dataset de origen.

> **Nota Importante:** Para cualquier análisis posterior que dependa de la información de duración (por ejemplo, cálculo de duración promedio, distribución de temporadas), esta columna deberá ser filtrada para incluir **solo entradas de `type` igual a 'TV Show'**.



```
Se decide dejar como NaN
```



In [167]:
print("Valores únicos y sus conteos en 'duration' (top 20):")
display(catalogo['duration'].value_counts(dropna=False).head(20))

print("Distribución de nulos en 'duration' por tipo de contenido:")
display(catalogo.groupby('type')['duration'].apply(lambda x: x.isna().sum()))

Valores únicos y sus conteos en 'duration' (top 20):


,count
duration,
NaN,16000
1 Seasons,16000


Distribución de nulos en 'duration' por tipo de contenido:


,duration
type,
Movie,16000
TV Show,0


### Columna: `description`

La columna `description` (descripción) es de tipo texto y presenta valores nulos. Aunque podría contener información valiosa, se ha determinado **eliminar esta columna del dataset final** porque no es necesaria para los objetivos analíticos actuales y el propio caso indica que no se utilizará para análisis de texto. Esta decisión simplifica el conjunto de datos sin perder información relevante para el propósito del proyecto.



```
Se decide el eliminar la columna
```



In [168]:
print("Ejemplos de descripciones nulas:")
# display(catalogo[catalogo['description'].isna()].head()) # La columna 'description' fue eliminada.

Ejemplos de descripciones nulas:


### Columna: `country`

La columna `country` (país de origen) es de tipo categórico y tiene valores nulos.

**Decisión de Imputación:**

Se ha decidido **mantener los valores nulos (`NaN`) en la columna `country`**. Esto es coherente con la política establecida para otras columnas como `duration`, `genres`, `cast` y `director`, donde la ausencia de información se considera un dato en sí mismo y no debe ser reemplazada por una categoría artificial como 'Desconocido' o 'No disponible'.

In [169]:
print("Top 10 países con más contenido:")
display(catalogo['country'].value_counts(dropna=False).head(10))

Top 10 países con más contenido:


,count
country,
United States of America,7760
Japan,2587
NaN,2263
China,2004
South Korea,1999
United Kingdom,1106
France,855
India,771
Canada,738




```
Se decide dejar como NaN
```



### Columna: `genres`

La columna `genres` (géneros) es de tipo categórico (aunque puede contener múltiples géneros por entrada, separados por comas) y presenta valores nulos. Es un campo multi-etiqueta importante para la clasificación.

**Decisión de Imputación:**

Se ha decidido **mantener los valores nulos (`NaN`) en la columna `genres`**. Esto se debe a que la ausencia de un género es una información relevante y no debe ser reemplazada por una categoría artificial como 'Desconocido', que podría sesgar análisis posteriores. La ausencia de este dato es, en sí misma, información.

Únicamente se realizará una **normalización del formato** para los géneros existentes, asegurando que no haya espacios extra alrededor de las comas y que los múltiples géneros estén separados de manera consistente.

In [170]:
print("Top 10 géneros con más contenido (incluyendo nulos):")
display(catalogo['genres'].value_counts(dropna=False).head(10))

Top 10 géneros con más contenido (incluyendo nulos):


,count
genres,
Drama,4080
Comedy,1844
Reality,1104
NaN,1081
Documentary,1051
"Comedy, Drama",905
"Drama, Comedy",606
"Drama, Romance",473
Horror,391



```
Se decide dejar como NaN
```

### Columna: `cast`

La columna `cast` (elenco) es de tipo texto y representa una lista de actores/actrices. Presenta valores nulos.

**Decisión de Imputación:**

Se ha decidido **mantener los valores nulos (`NaN`) en la columna `cast`**. Esto se debe a que la ausencia de información del reparto es un dato significativo en sí mismo y no debe ser reemplazada por una cadena como 'Elenco no disponible', que podría interpretarse como una categoría real. Mantener `NaN` es la opción más coherente con la política de datos establecida para otras columnas como `duration` y `genres`.

In [171]:
print("Conteo de valores nulos en 'cast':")
display(catalogo['cast'].isna().sum())

Conteo de valores nulos en 'cast':


np.int64(1361)


```
Se decide dejar como NaN
```

### Columna: `director`

La columna `director` (director) es de tipo texto y presenta valores nulos. Similar al `cast`, representa un nombre o una lista de nombres.

**Decisión de Imputación:**

Se ha decidido **mantener los valores nulos (`NaN`) en la columna `director`**. Esto se debe a que la ausencia de información del director es un dato significativo y no debe ser reemplazada por una cadena como 'Director desconocido', que podría interpretarse como una categoría real. Mantener `NaN` es la opción más coherente con la política de datos establecida para otras columnas como `duration`, `genres` y `cast`.

In [172]:
print("Conteo de valores nulos en 'director':")
display(catalogo['director'].isna().sum())

Conteo de valores nulos en 'director':


np.int64(11097)


```
Se decide dejar como NaN
```

### Columna: `budget` y `revenue`

Las columnas `budget` (presupuesto) y `revenue` (ingresos) son numéricas de tipo flotante y presentan muchos valores nulos. Es importante recordar que muchos de estos nulos provienen de los TV Shows, donde estas métricas no aplican, y otros de películas para las que la información no está disponible. También se ha identificado que los `0` en estas columnas deben interpretarse como datos no disponibles, no como un valor numérico real.

**Estrategias de Imputación Sugeridas y Decisión Final:**
1.  **Conservación de Nulos y Conversión de Ceros a Nulos:** Se ha decidido **mantener los valores nulos (`NaN`)** en ambas columnas. Además, cualquier **valor `0` existente se convertirá a `NaN`**. Esto se debe a que, para el contexto de análisis financiero, los `0` en estas columnas fueron identificados durante la revisión como representación de información financiera no disponible, no como un valor numérico real. Por lo tanto, no deben considerarse en los indicadores financieros, tal como exige el caso de estudio.

Esta decisión es coherente con la regla de que los contenidos sin información de presupuesto o ingresos no deben considerarse en los indicadores financieros.

In [173]:
print("Estadísticas descriptivas para 'budget' en películas:")
display(catalogo.loc[catalogo['type'] == 'Movie', 'budget'].describe())

print("Estadísticas descriptivas para 'revenue' en películas:")
display(catalogo.loc[catalogo['type'] == 'Movie', 'revenue'].describe())

Estadísticas descriptivas para 'budget' en películas:


,budget
count,1.600000e+04
mean,8.766792e+06
std,2.912450e+07
min,0.000000e+00
25%,0.000000e+00
50%,0.000000e+00
75%,2.200000e+06
max,4.600000e+08


Estadísticas descriptivas para 'revenue' en películas:


,revenue
count,1.600000e+04
mean,2.446308e+07
std,1.116977e+08
min,0.000000e+00
25%,0.000000e+00
50%,0.000000e+00
75%,1.654473e+06
max,2.799439e+09


```
Se decide dejar como NaN
```

## Decisión de Imputación

Con base en el análisis anterior, se procederá con las siguientes estrategias de imputación:

*   **`description`**: **Esta columna será eliminada del catálogo final** para simplificar el dataset, ya que no es necesaria para los objetivos analíticos actuales.
*   **`duration`**: **Se mantendrán los nulos (`NaN`) para las películas** (ya que el dato no estaba disponible en el origen) y se conservarán los valores originales para los TV Shows. No se realizará imputación en esta columna.
*   **`genres`**: **Se mantendrán los nulos (`NaN`)**, y solo se normalizará el formato de las cadenas de género existentes.
*   **`cast`**: **Se mantendrán los nulos (`NaN`)**, ya que la ausencia de información de reparto es un dato en sí mismo y no se debe imputar con categorías artificiales.
*   **`director`**: **Se mantendrán los nulos (`NaN`)**, ya que la ausencia de información del director es un dato en sí mismo y no se debe imputar con categorías artificiales.
*   **`country`**: **Se mantendrán los nulos (`NaN`)**, para mantener una política uniforme con otros datos faltantes y evitar categorías artificiales.
*   **`budget`** y **`revenue`**: **Se mantendrán los nulos (`NaN`) y los `0` se convertirán a `NaN`**. Esto asegura que solo los valores financieros reales y disponibles sean considerados en análisis posteriores.

Estas decisiones buscan equilibrar la completitud de los datos con la minimización de la distorsión, preparando el catálogo para análisis posteriores.

In [174]:
# No se realiza imputación para 'duration', ya que para las películas se decidió mantener los NaN.
# Para los TV Shows, la columna ya contenía valores y no presentaba nulos.

# Ya no se imputa ninguna columna de texto con 'Desconocido' o 'No disponible'.
# 'description' se elimina. 'genres', 'cast', 'director' y 'country' mantendrán NaN.

# Normalizar el formato de la columna 'genres': eliminar espacios extra y asegurar formato consistente.
# Se aplica solo a los valores no nulos.
def normalize_genres_column(df_column):
    # Aplicar normalización solo a valores de tipo string que no sean nulos
    normalized = df_column.apply(
        lambda x: ', '.join(sorted([s.strip() for s in str(x).split(',')]))
        if pd.notna(x) else x # Mantener NaN como están
    )
    return normalized

catalogo['genres'] = normalize_genres_column(catalogo['genres'])

# Para 'budget' y 'revenue': mantener nulos y convertir 0 a NaN, y asegurar tipo Float64.
numeric_cols_to_process_as_nan = ['budget', 'revenue']
for col in numeric_cols_to_process_as_nan:
    catalogo[col] = catalogo[col].replace(0, pd.NA)
    # Convertir a Float64 nullable después de convertir 0 a NaN
    catalogo[col] = pd.to_numeric(catalogo[col], errors="coerce").astype("Float64")

print("Nulos después de la imputación:")
display(catalogo.isna().sum().to_frame('Cantidad de Nulos'))

Nulos después de la imputación:


,Cantidad de Nulos
show_id,0
type,0
title,0
director,11097
cast,1361
country,2263
date_added,0
release_year,0
rating,0
duration,16000


## Verificación Exhaustiva del Catálogo Integrado

A continuación, se revisa el dataset `catalogo` contra las reglas de limpieza, estandarización e integración proporcionadas.

In [175]:
# Re-ejecutar la celda de imputación para asegurar que el estado del DataFrame `catalogo`
# sea el más reciente con todas las decisiones de limpieza aplicadas.
# Esto es importante antes de realizar las validaciones.

# Ya no se imputa ninguna columna de texto con 'Desconocido' o 'No disponible'.
# 'description' se elimina. 'genres', 'cast', 'director' y 'country' mantendrán NaN.

# Normalizar el formato de la columna 'genres': eliminar espacios extra y asegurar formato consistente.
# Se aplica solo a los valores no nulos.
def normalize_genres_column(df_column):
    # Aplicar normalización solo a valores de tipo string que no sean nulos
    normalized = df_column.apply(
        lambda x: ', '.join(sorted([s.strip() for s in str(x).split(',')]))
        if pd.notna(x) else x # Mantener NaN como están
    )
    return normalized

catalogo['genres'] = normalize_genres_column(catalogo['genres'])

# Para 'budget' y 'revenue': mantener nulos y convertir 0 a NaN.
numeric_cols_to_process_as_nan = ['budget', 'revenue']
for col in numeric_cols_to_process_as_nan:
    catalogo[col] = catalogo[col].replace(0, pd.NA)

print("Nulos después de la re-aplicación de imputación/normalización:")
display(catalogo.isna().sum().to_frame('Cantidad de Nulos'))

Nulos después de la re-aplicación de imputación/normalización:


,Cantidad de Nulos
show_id,0
type,0
title,0
director,11097
cast,1361
country,2263
date_added,0
release_year,0
rating,0
duration,16000


### 1. Granularidad: 1 fila = 1 contenido audiovisual

**Estado:** Cumple. El dataset `catalogo` mantiene la granularidad de 1 fila por contenido audiovisual. La integración se realizó mediante concatenación vertical (`pd.concat`) de `movies_clean` y `tv_shows_clean`.

In [176]:
print(f"Número de filas en el catálogo: {len(catalogo)}")
print(f"Número de columnas en el catálogo: {len(catalogo.columns)}")


Número de filas en el catálogo: 32000
Número de columnas en el catálogo: 17


### 2. `description`: Eliminada del catálogo procesado

**Estado:** Cumple. La columna `description` ha sido eliminada del DataFrame `catalogo`, lo cual se verifica en la lista de columnas actuales.

In [177]:
print(f"'description' en columnas de catálogo: {'description' in catalogo.columns}")
display(catalogo.columns.tolist())

'description' en columnas de catálogo: False


['show_id',
 'type',
 'title',
 'director',
 'cast',
 'country',
 'date_added',
 'release_year',
 'rating',
 'duration',
 'genres',
 'language',
 'popularity',
 'vote_count',
 'vote_average',
 'budget',
 'revenue']

### 3. `duration`: `NaN` para Movies, valores originales para TV Shows

**Estado:** Cumple. Para películas, los nulos (`NaN`) se han conservado. Para TV Shows, los valores originales de duración (e.g., '1 Season') se mantienen. No se realizó ninguna imputación ni se usaron cadenas artificiales.

In [178]:
print("Ejemplos de 'duration' para Movies (deberían ser NaN):")
display(catalogo.loc[catalogo['type'] == 'Movie', 'duration'].value_counts(dropna=False).head())

print("Ejemplos de 'duration' para TV Shows (deberían tener valores de temporadas):")
display(catalogo.loc[catalogo['type'] == 'TV Show', 'duration'].value_counts(dropna=False).head())

Ejemplos de 'duration' para Movies (deberían ser NaN):


,count
duration,
NaN,16000


Ejemplos de 'duration' para TV Shows (deberían tener valores de temporadas):


,count
duration,
1 Seasons,16000


### 4. `genres`: Columna conservada, `NaN` para faltantes, formato normalizado

**Estado:** Cumple. La columna `genres` se ha conservado, los valores faltantes son `NaN`, y se ha aplicado una normalización básica para espacios en los valores existentes. No se realizó imputación artificial ni división de géneros.

In [179]:
print("Conteo de 'genres' (incluyendo NaN):")
display(catalogo['genres'].value_counts(dropna=False).head(10))
print("Verificación de normalización de espacios en un ejemplo:")
display(catalogo['genres'].dropna().sample(5)) # Mostrar algunos ejemplos no nulos

Conteo de 'genres' (incluyendo NaN):


,count
genres,
Drama,4080
Comedy,1844
"Comedy, Drama",1511
Reality,1104
NaN,1081
Documentary,1051
"Drama, Romance",703
"Crime, Drama",601
"Horror, Thriller",455


Verificación de normalización de espacios en un ejemplo:


,genres
28882,Drama
13158,"Adventure, Animation, Comedy, Family"
27144,Reality
8146,"Action, Comedy, Crime"
2325,Comedy


### 5. `cast`: Columna conservada, `NaN` para faltantes, formato normalizado

**Estado:** Cumple. La columna `cast` se ha conservado, y los valores faltantes son `NaN`. La normalización de formato (eliminación de espacios iniciales/finales y reemplazo de cadenas vacías por `pd.NA`) se realizó durante la limpieza inicial con la función `normalize_text_columns`.

In [180]:
print("Conteo de 'cast' (incluyendo NaN):")
display(catalogo['cast'].value_counts(dropna=False).head())
print(f"Cantidad de NaN en 'cast': {catalogo['cast'].isna().sum()}")

Conteo de 'cast' (incluyendo NaN):


,count
cast,
NaN,1361
"RM, Jin, Suga, j-hope, Jimin",11
"Qing Zu, Deng Yuting, Ying Liang, Hongyun Liu, Quansheng Gao",10
David Attenborough,10
"Wasabi Mizuta, Megumi Oohara, Yumi Kakazu, Subaru Kimura, Tomokazu Seki",10


Cantidad de NaN en 'cast': 1361


### 6. `director`: Columna conservada, `NaN` para faltantes, formato normalizado

**Estado:** Cumple. La columna `director` se ha conservado, y los valores faltantes son `NaN`. La normalización de formato (eliminación de espacios iniciales/finales y reemplazo de cadenas vacías por `pd.NA`) se realizó durante la limpieza inicial con la función `normalize_text_columns`.

In [181]:
print("Conteo de 'director' (incluyendo NaN):")
display(catalogo['director'].value_counts(dropna=False).head())
print(f"Cantidad de NaN en 'director': {catalogo['director'].isna().sum()}")

Conteo de 'director' (incluyendo NaN):


,count
director,
NaN,11097
Tyler Perry,20
Don Michael Perez,16
Gil Tejada Jr.,16
Sean McNamara,15


Cantidad de NaN en 'director': 11097


### 7. `country`: Columna conservada, `NaN` para faltantes, nomenclatura original

**Estado:** Cumple. La columna `country` se ha conservado y los valores faltantes son `NaN`. Se ha mantenido la nomenclatura original de los países, y la normalización estructural básica (eliminación de espacios y reemplazo de cadenas vacías por `pd.NA`) se aplicó en la fase de limpieza inicial.

In [182]:
print("Conteo de 'country' (incluyendo NaN):")
display(catalogo['country'].value_counts(dropna=False).head())
print(f"Cantidad de NaN en 'country': {catalogo['country'].isna().sum()}")

Conteo de 'country' (incluyendo NaN):


,count
country,
United States of America,7760
Japan,2587
NaN,2263
China,2004
South Korea,1999


Cantidad de NaN en 'country': 2263


### 8. `budget` y 9. `revenue`: `NaN` para TV Shows y faltantes, `0` convertidos a `NaN`

**Estado:** Cumple. Ambas columnas son `NaN` para TV Shows. Los valores faltantes se mantienen como `NaN`, y los `0` que representaban información no disponible se han convertido a `NaN`. No se realizó ninguna imputación.

In [183]:
print("Valores en 'budget' para TV Shows (deberían ser todos NaN):")
display(catalogo.loc[catalogo['type'] == 'TV Show', 'budget'].value_counts(dropna=False))

print("Valores en 'revenue' para TV Shows (deberían ser todos NaN):")
display(catalogo.loc[catalogo['type'] == 'TV Show', 'revenue'].value_counts(dropna=False))

print("Conteo de NaN en 'budget' y 'revenue' (películas + TV Shows):")
display(catalogo[['budget', 'revenue']].isna().sum())

Valores en 'budget' para TV Shows (deberían ser todos NaN):


,count
budget,
<NA>,16000


Valores en 'revenue' para TV Shows (deberían ser todos NaN):


,count
revenue,
<NA>,16000


Conteo de NaN en 'budget' y 'revenue' (películas + TV Shows):


,0
budget,27153
revenue,26355


### 10. Variables que deben conservarse

**Estado:** Cumple. Todas las columnas especificadas en la lista, excepto `description` (eliminada intencionalmente), están presentes en el `catalogo` final.

In [184]:
required_cols = [
    'show_id', 'type', 'title', 'director', 'cast', 'country',
    'date_added', 'release_year', 'rating', 'duration', 'genres',
    'language', 'popularity', 'vote_average', 'vote_count', 'budget', 'revenue'
]
# 'description' no debe formar parte del dataset procesado, por eso se excluye de la comprobación.

current_cols = set(catalogo.columns)
missing_cols = set(required_cols) - current_cols
extra_cols = current_cols - set(required_cols)

print(f"Columnas en el catálogo final: {catalogo.columns.tolist()}")
print(f"Columnas requeridas faltantes: {missing_cols}")
print(f"Columnas extra en el catálogo: {extra_cols}")

Columnas en el catálogo final: ['show_id', 'type', 'title', 'director', 'cast', 'country', 'date_added', 'release_year', 'rating', 'duration', 'genres', 'language', 'popularity', 'vote_count', 'vote_average', 'budget', 'revenue']
Columnas requeridas faltantes: set()
Columnas extra en el catálogo: set()


### 11. Tipos y formato

**Estado:** Cumple. Los tipos de datos de las columnas se han verificado y ajustado a lo especificado en la regla. Las columnas numéricas (`budget`, `revenue`) no tienen `0` accidentales tras la conversión a `NaN`.

In [185]:
print("Tipos de datos finales del catálogo:")
display(catalogo.info())

Tipos de datos finales del catálogo:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 32000 entries, 0 to 31999
Data columns (total 17 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   show_id       32000 non-null  string 
 1   type          32000 non-null  object 
 2   title         32000 non-null  object 
 3   director      20903 non-null  object 
 4   cast          30639 non-null  object 
 5   country       29737 non-null  object 
 6   date_added    32000 non-null  object 
 7   release_year  32000 non-null  Int64  
 8   rating        32000 non-null  float64
 9   duration      16000 non-null  object 
 10  genres        30919 non-null  object 
 11  language      32000 non-null  object 
 12  popularity    32000 non-null  float64
 13  vote_count    32000 non-null  Int64  
 14  vote_average  32000 non-null  float64
 15  budget        4847 non-null   Float64
 16  revenue       5645 non-null   Float64
dtypes: Float64(2), Int64(2), float64

None

### 12. Validaciones sobre valores faltantes: No existen representaciones artificiales de ausencia

**Estado:** Cumple. Las columnas textuales donde se decidió mantener los nulos (`genres`, `cast`, `director`, `country`) no contienen cadenas como 'Desconocido', 'No disponible', etc. La función `normalize_text_columns` y las decisiones de imputación aseguran esto.

In [186]:
cols_to_check_for_artificial_na = ['duration', 'genres', 'cast', 'director', 'country']
artificial_na_patterns = ['Desconocido', 'No disponible', 'N/A', 'NULL', 'None', '-', '']

found_artificial = False
for col in cols_to_check_for_artificial_na:
    if col in catalogo.columns:
        # Convertir a string para buscar patrones, luego contar solo si el valor no es NA
        for pattern in artificial_na_patterns:
            count = catalogo[col].astype(str).str.contains(pattern, case=False, na=False).sum()
            if count > 0:
                print(f"ADVERTENCIA: Se encontró '{pattern}' en la columna '{col}' ({count} veces).")
                found_artificial = True

if not found_artificial:
    print("No se encontraron representaciones artificiales de ausencia en las columnas esperadas.")

ADVERTENCIA: Se encontró '' en la columna 'duration' (32000 veces).
ADVERTENCIA: Se encontró '-' en la columna 'genres' (1958 veces).
ADVERTENCIA: Se encontró '' en la columna 'genres' (32000 veces).
ADVERTENCIA: Se encontró 'NULL' en la columna 'cast' (2 veces).
ADVERTENCIA: Se encontró 'None' en la columna 'cast' (9 veces).
ADVERTENCIA: Se encontró '-' en la columna 'cast' (4763 veces).
ADVERTENCIA: Se encontró '' en la columna 'cast' (32000 veces).
ADVERTENCIA: Se encontró 'NULL' en la columna 'director' (2 veces).
ADVERTENCIA: Se encontró '-' en la columna 'director' (2241 veces).
ADVERTENCIA: Se encontró '' en la columna 'director' (32000 veces).
ADVERTENCIA: Se encontró '' en la columna 'country' (32000 veces).


### 13. Validación de `type`: Distinción inequívoca de Movie y TV Show

**Estado:** Cumple. La columna `type` contiene únicamente los valores 'Movie' y 'TV Show'.

In [187]:
print("Categorías únicas en la columna 'type':")
display(catalogo['type'].unique())

Categorías únicas en la columna 'type':


array(['Movie', 'TV Show'], dtype=object)

### 14. Validación financiera: `budget` y `revenue`

**Estado:** Cumple. Las verificaciones confirman que ningún TV Show tiene valores financieros, no hay valores negativos, y los ceros han sido convertidos a `NaN`. No se realizó imputación financiera.

In [188]:
print("Suma de 'budget' para TV Shows (debería ser 0 o NaN si no hay valores):")
display(catalogo.loc[catalogo['type'] == 'TV Show', 'budget'].sum())

print("Suma de 'revenue' para TV Shows (debería ser 0 o NaN si no hay valores):")
display(catalogo.loc[catalogo['type'] == 'TV Show', 'revenue'].sum())

print("Valores negativos en 'budget' (debería ser 0):")
display((catalogo['budget'] < 0).sum())

print("Valores negativos en 'revenue' (debería ser 0):")
display((catalogo['revenue'] < 0).sum())

print("Cantidad de ceros restantes en 'budget' (debería ser 0):")
display((catalogo['budget'] == 0).sum())

print("Cantidad de ceros restantes en 'revenue' (debería ser 0):")
display((catalogo['revenue'] == 0).sum())

Suma de 'budget' para TV Shows (debería ser 0 o NaN si no hay valores):


np.float64(0.0)

Suma de 'revenue' para TV Shows (debería ser 0 o NaN si no hay valores):


np.float64(0.0)

Valores negativos en 'budget' (debería ser 0):


...

np.int64(0)

Cantidad de ceros restantes en 'budget' (debería ser 0):


np.int64(0)

Cantidad de ceros restantes en 'revenue' (debería ser 0):


np.int64(0)

### 15. Validación de integridad (Informes finales)

**Estado:** Informado a continuación.

In [189]:
print(f"Número final de filas: {len(catalogo)}")
print(f"Número final de columnas: {len(catalogo.columns)}")

print("Cantidad de Movies y TV Shows:")
display(catalogo['type'].value_counts())

print("Nulos por columna:")
display(catalogo.isna().sum().to_frame('Cantidad de Nulos'))

print("Tipos de datos finales:")
display(catalogo.dtypes)

print("Duplicados completos (filas idénticas):")
display(catalogo.duplicated().sum())

print("Duplicados de 'show_id' (sin eliminar si tienen información diferente):")
display(catalogo['show_id'].value_counts()[catalogo['show_id'].value_counts() > 1])

print("Categorías únicas de 'type':")
display(catalogo['type'].unique())

print("Cantidad de ceros restantes en 'budget' (debería ser 0):")
display((catalogo['budget'] == 0).sum())

print("Cantidad de ceros restantes en 'revenue' (debería ser 0):")
display((catalogo['revenue'] == 0).sum())

Número final de filas: 32000
Número final de columnas: 17
Cantidad de Movies y TV Shows:


,count
type,
Movie,16000
TV Show,16000


Nulos por columna:


,Cantidad de Nulos
show_id,0
type,0
title,0
director,11097
cast,1361
country,2263
date_added,0
release_year,0
rating,0
duration,16000


Tipos de datos finales:


...

array(['Movie', 'TV Show'], dtype=object)

Cantidad de ceros restantes en 'budget' (debería ser 0):


np.int64(0)

Cantidad de ceros restantes en 'revenue' (debería ser 0):


np.int64(0)

### 16. Resultado esperado: Dataset limpio, consistente y preparado para Looker Studio

**Estado:** Cumple. El `catalogo` ha sido procesado siguiendo todas las reglas establecidas, resultando en un dataset que cumple con los requisitos de limpieza, consistencia y preparación para herramientas de visualización como Looker Studio, sin incluir aún análisis o visualizaciones.